In [ ]:
# =============================================================================
# CASE STUDY: MULTIPLE LINEAR REGRESSION — PREDICTION FRAMEWORK
# Employee salary (k$)
# =============================================================================
# Variables:
#   Y  : Annual salary (k$)                              [response]
#   X1 : Years of work experience
#   X2 : Education level (1=Technical, 2=Professional, 3=Postgraduate)
#   X3 : Department area (1=Tech, 0=Non-tech)            [dummy]
#   X4 : Seniority flag (1=Senior, 0=Junior)             [dummy]
#   X5 : Professional certifications (1=Yes, 0=No)       [dummy]
# -----------------------------------------------------------------------------
# Target transformation:
#   log(Y) is used as the response variable. Salary has a multiplicative
#   structure — each predictor shifts salary by a percentage, not a fixed
#   amount — so the raw scale exhibits heteroscedasticity and right skew.
#   (As exercise check the previous statement)
#   All error metrics (RMSE, MAE) are back-transformed to the original k$ scale.
# -----------------------------------------------------------------------------
# Approach:
#   - 80/20 train/test split (random_state = 42)
#   - Forward selection   : add the variable that most reduces test RMSE
#   - Backward elimination: remove the variable that most reduces test RMSE
#   - Stopping rule       : stop when no step improves test RMSE
#   - Final comparison    : full model vs forward model vs backward model
# =============================================================================


import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# =============================================================================
# 1. LOAD & PREPARE
# =============================================================================

df = pd.read_csv('Class 5 - Salary.csv')
print(df.head())

FEATURES = ["experience", "education", "area", "senior", "certif"]
TARGET   = "log_salary"                  # log-transform applied beforehand
df[TARGET] = np.log(df["salary"])      

X = df[FEATURES].values
y_log = df[TARGET].values
y_raw = df["salary"].values

X_tr, X_te, yl_tr, yl_te, yr_te = [None]*5
X_tr, X_te, yl_tr, yl_te = train_test_split(X, y_log, test_size=0.20, random_state=42)
yr_te = np.exp(yl_te)                    # test labels in original salary scale

print(f"\nTrain: {len(X_tr)} obs  |  Test: {len(X_te)} obs")

# =============================================================================
# 2. METRICS  (always reported in original salary scale)
# =============================================================================

def test_errors(col_idx):
    """Fit on train, predict on test, return RMSE and MAE in salary scale."""
    model  = LinearRegression().fit(X_tr[:, col_idx], yl_tr)
    y_pred = np.exp(model.predict(X_te[:, col_idx]))
    rmse   = np.sqrt(mean_squared_error(yr_te, y_pred))
    mae    = mean_absolute_error(yr_te, y_pred)
    return rmse, mae

   experience  education  area  senior  certif  salary  log_salary
0           7          2     1       0       1  134.21    4.899413
1          20          2     0       1       0  469.73    6.152167
2          15          3     0       1       0  282.04    5.642047
3          11          2     1       1       0  226.65    5.423397
4           8          1     0       1       1   97.11    4.575881

Train: 800 obs  |  Test: 200 obs


In [2]:

# =============================================================================
# 3. FORWARD SELECTION  (add variable that most reduces test RMSE)
# =============================================================================

print("\n" + "=" * 60)
print("FORWARD SELECTION  —  criterion: test RMSE")
print("=" * 60)
print(f"{'Step':<6} {'Added':<20} {'Test RMSE':>10} {'Test MAE':>10}")
print("-" * 48)

selected_fw = []
remaining   = list(range(len(FEATURES)))

while remaining:
    best_rmse, best_var = np.inf, None
    for i in remaining:
        rmse, _ = test_errors(selected_fw + [i])
        if rmse < best_rmse:
            best_rmse, best_var = rmse, i
    # stop if adding the best candidate does not improve RMSE
    if selected_fw:
        current_rmse, _ = test_errors(selected_fw)
        if best_rmse >= current_rmse:
            break
    selected_fw.append(best_var)
    remaining.remove(best_var)
    rmse_fw, mae_fw = test_errors(selected_fw)
    print(f"{len(selected_fw):<6} {FEATURES[best_var]:<20} {rmse_fw:>10.2f} {mae_fw:>10.2f}")

print(f"\nForward model : {[FEATURES[i] for i in selected_fw]}")




FORWARD SELECTION  —  criterion: test RMSE
Step   Added                 Test RMSE   Test MAE
------------------------------------------------
1      experience                91.08      61.86
2      education                 73.76      47.38
3      senior                    70.48      45.70
4      area                      67.26      39.98
5      certif                    65.59      39.11

Forward model : ['experience', 'education', 'senior', 'area', 'certif']


In [3]:
# =============================================================================
# 4. BACKWARD ELIMINATION  (remove variable that most reduces test RMSE)
# =============================================================================

print("\n" + "=" * 60)
print("BACKWARD ELIMINATION  —  criterion: test RMSE")
print("=" * 60)
print(f"{'Step':<6} {'Removed':<20} {'Test RMSE':>10} {'Test MAE':>10}")
print("-" * 48)

selected_bw = list(range(len(FEATURES)))

while len(selected_bw) > 1:
    current_rmse, _ = test_errors(selected_bw)
    best_rmse, worst_var = np.inf, None
    for i in selected_bw:
        candidate = [j for j in selected_bw if j != i]
        rmse, _   = test_errors(candidate)
        if rmse < best_rmse:
            best_rmse, worst_var = rmse, i
    # stop if removing any variable does not improve RMSE
    if best_rmse >= current_rmse:
        break
    selected_bw.remove(worst_var)
    rmse_bw, mae_bw = test_errors(selected_bw)
    print(f"{len(selected_bw):<6} {FEATURES[worst_var]:<20} {rmse_bw:>10.2f} {mae_bw:>10.2f}")

print(f"\nBackward model: {[FEATURES[i] for i in selected_bw]}")




BACKWARD ELIMINATION  —  criterion: test RMSE
Step   Removed               Test RMSE   Test MAE
------------------------------------------------

Backward model: ['experience', 'education', 'area', 'senior', 'certif']


In [4]:
# =============================================================================
# 5. SUMMARY
# =============================================================================

print("\n" + "=" * 60)
print("FINAL COMPARISON")
print("=" * 60)

full_idx = list(range(len(FEATURES)))
rows = [
    ("Full model",      full_idx),
    ("Forward model",   selected_fw),
    ("Backward model",  selected_bw),
]

print(f"\n{'Model':<22} {'Features':<35} {'Test RMSE':>10} {'Test MAE':>9}")
print("-" * 78)
for label, idx in rows:
    r, m = test_errors(idx)
    feat = ", ".join(FEATURES[i] for i in idx)
    print(f"{label:<22} {feat:<35} {r:>10.2f} {m:>9.2f}")


FINAL COMPARISON

Model                  Features                             Test RMSE  Test MAE
------------------------------------------------------------------------------
Full model             experience, education, area, senior, certif      65.59     39.11
Forward model          experience, education, senior, area, certif      65.59     39.11
Backward model         experience, education, area, senior, certif      65.59     39.11


In [5]:
# =============================================================================
# 6. AIC AND BIC — IN-SAMPLE PENALIZED CRITERIA
# -----------------------------------------------------------------------------
# Both criteria penalize model complexity to discourage overfitting.
# They are computed on the TRAINING SET using the log-likelihood of the model.
#
# Formulas (for OLS with normal errors, fitted on n observations with k
# predictors plus intercept, i.e. p = k + 1 parameters):
#
#   SSR  = sum of squared residuals  (training)
#   σ²   = SSR / n                   (MLE estimate of error variance)
#   L    = log-likelihood = -n/2 * log(2π σ²) - SSR / (2σ²)
#                         = -n/2 * [log(2π) + log(SSR/n) + 1]
#
#   AIC  = -2·L + 2·p      →  AIC = n·log(SSR/n) + 2·p   (+ const)
#   BIC  = -2·L + log(n)·p →  BIC = n·log(SSR/n) + log(n)·p
#
# Lower AIC / BIC = better model.
# BIC penalizes extra parameters more heavily when n is large (log(n) > 2),
# so it tends to select smaller models than AIC.
#
# Key conceptual difference vs test RMSE:
#   - AIC / BIC use only training data; they approximate out-of-sample error
#     through the penalty term, without actually holding out observations.
#   - Test RMSE measures error directly on unseen data.
#   Comparing both reveals whether the penalty is a good proxy for
#   true generalization.
# =============================================================================

import statsmodels.api as sm

def aic_bic(col_idx):
    """Fit OLS on training set, return AIC and BIC (statsmodels convention)."""
    X_sm = sm.add_constant(X_tr[:, col_idx])
    res  = sm.OLS(yl_tr, X_sm).fit()
    return res.aic, res.bic

# --- AIC/BIC for every candidate model ---

full_idx = list(range(len(FEATURES)))
rows_ic  = [
    ("Full model",      full_idx),
    ("Forward model",   selected_fw),
    ("Backward model",  selected_bw),
]

print("\n" + "=" * 60)
print("AIC AND BIC — penalized in-sample criteria (train set)")
print("=" * 60)
print(f"\n{'Model':<22} {'k':>3} {'AIC':>10} {'BIC':>10} {'Test RMSE':>10}")
print("-" * 57)
for label, idx in rows_ic:
    aic, bic   = aic_bic(idx)
    rmse_t, _  = test_errors(idx)
    print(f"{label:<22} {len(idx):>3} {aic:>10.2f} {bic:>10.2f} {rmse_t:>10.2f}")

# --- All-subsets AIC/BIC table (same 31 subsets as before) ---

print("\n" + "=" * 60)
print("ALL SUBSETS — AIC, BIC and test RMSE ranked by AIC")
print("=" * 60)
print(f"\n{'Rank':<5} {'k':>2}  {'Subset':<38} {'AIC':>8} {'BIC':>8} {'Test RMSE':>10}")
print("-" * 75)

subset_rows = []
for k in range(1, len(FEATURES) + 1):
    for combo in combinations(range(len(FEATURES)), k):
        idx        = list(combo)
        aic, bic   = aic_bic(idx)
        rmse_t, _  = test_errors(idx)
        subset_rows.append({
            "label"    : " + ".join(FEATURES[i] for i in idx),
            "k"        : k,
            "idx"      : idx,
            "aic"      : aic,
            "bic"      : bic,
            "test_rmse": rmse_t,
        })

subset_df = pd.DataFrame(subset_rows).sort_values("aic").reset_index(drop=True)

for rank, row in subset_df.iterrows():
    marker = "  <- best AIC" if rank == 0 else ""
    print(f"{rank+1:<5} {row['k']:>2}  {row['label']:<38} "
          f"{row['aic']:>8.2f} {row['bic']:>8.2f} {row['test_rmse']:>10.2f}{marker}")

# --- Best model per criterion ---

best_aic  = subset_df.loc[subset_df["aic"].idxmin()]
best_bic  = subset_df.loc[subset_df["bic"].idxmin()]
best_rmse = subset_df.loc[subset_df["test_rmse"].idxmin()]

print("\n" + "=" * 60)
print("WINNER PER CRITERION")
print("=" * 60)
print(f"\n{'Criterion':<14} {'Selected features':<42} {'Value':>8}")
print("-" * 66)
print(f"{'AIC':<14} {best_aic['label']:<42} {best_aic['aic']:>8.2f}")
print(f"{'BIC':<14} {best_bic['label']:<42} {best_bic['bic']:>8.2f}")
print(f"{'Test RMSE':<14} {best_rmse['label']:<42} {best_rmse['test_rmse']:>8.2f}")

agree = (best_aic["label"] == best_bic["label"] == best_rmse["label"])
print(f"\nAll three criteria agree: {agree}")
if not agree:
    print("→ Disagreement reveals the gap between in-sample penalties and")
    print("  true out-of-sample generalization. Test RMSE is the ground truth.")


AIC AND BIC — penalized in-sample criteria (train set)

Model                    k        AIC        BIC  Test RMSE
---------------------------------------------------------
Full model               5    -301.33    -273.22      65.59
Forward model            5    -301.33    -273.22      65.59
Backward model           5    -301.33    -273.22      65.59

ALL SUBSETS — AIC, BIC and test RMSE ranked by AIC

Rank   k  Subset                                      AIC      BIC  Test RMSE
---------------------------------------------------------------------------
1      5  experience + education + area + senior + certif  -301.33  -273.22      65.59  <- best AIC
2      4  experience + education + area + senior  -282.19  -258.77      67.26
3      4  experience + education + area + certif  -255.31  -231.89      69.11
4      3  experience + education + area           -234.53  -215.79      70.78
5      4  experience + education + senior + certif   -57.08   -33.66      69.92
6      3  experience + e